In [ ]:
import torch
import torch.nn as nn

levels = torch.linspace(-1, 1, 5)
x = torch.randint(low=-100_000_000, high=100_000_000, size=(10, 1)).to(torch.float32) / 50_000_000
x = torch.tanh(x)
distances = torch.abs(levels - x)
best_matching_index = distances.sort().indices[0]
best_matching_index

In [ ]:
class Quantizer(nn.Module):
    def __init__(self, n_levels):
        super().__init__()
        self.n_levels = n_levels
        self.register_buffer('levels', torch.linspace(-1, 1, n_levels))

    def forward(self, x):
        # x: [B, 1]
        x = torch.tanh(x)
        distances = torch.abs(self.levels - x)
        best_matching_index = distances.sort().indices[:, 0]
        x_quantized = self.levels[best_matching_index].unsqueeze(1)
        x_quantized_st = x + (x_quantized - x).detach()
        return x_quantized_st, best_matching_index.unsqueeze(1) # [B, 1]

x = torch.randint(low=-100_000_000, high=100_000_000, size=(10, 1)).to(torch.float32) / 50_000_000
quantizer = Quantizer(n_levels=5)
print(x.shape)
quantizer(x)

In [ ]:
torch.tensor([1, 2]) *torch.tensor([1, 2])

In [ ]:
class FSQ(nn.Module):
    def __init__(self, levels: list[int]):
        super().__init__()
        multiplier = []
        for index in range(len(levels)):
            sub_levels = levels[:index]
            if sub_levels is None:
                multiplier.append(torch.tensor([0], dtype=torch.long))
            multiplier.append(torch.prod(torch.tensor(sub_levels, dtype=torch.long)))
            
        self.register_buffer('multiplier', torch.tensor(multiplier))

        self.quantizers = []
        for level in levels:
            self.quantizers.append(Quantizer(level))

    def forward(self, x):
        # x: B, T, n_levels
        x_flatten = torch.flatten(x, start_dim=0, end_dim=1) # B * T, n_levels

        y = []
        z = []
        for i, quantizer in enumerate(self.quantizers):
            x_quantized_st, code = quantizer(x_flatten[:, i].unsqueeze(1))
            y.append(code)
            z.append(x_quantized_st)

        y = torch.stack(y).squeeze() # n_levels, B * T
        y = y.permute(1, 0) # B * T, n_levels
        y = y * self.multiplier # B * T, n_levels
        code = torch.sum(y, dim=1) # B * T
        z = torch.stack(z).squeeze()
        return z.reshape(x.shape), code.reshape(x.shape[0], x.shape[1])

x = torch.randint(low=-100_000_000, high=100_000_000, size=(2, 3, 5)).to(torch.float32) / 50_000_000
fsq = FSQ([5, 5, 3, 3, 3])
fsq(x)[0].shape

In [ ]:
import matplotlib.pyplot as plt

random_values = torch.randint(low=-100_000_000, high=100_000_000, size=(100,)).to(torch.float32) / 50_000_000

quantized_values = [fsq(value).item() for value in random_values]
values = [value.item() for value in random_values]

values_t = torch.tensor(values, dtype=torch.float32)
tanh_x = torch.tanh(values_t)
idx = torch.tensor(quantized_values, dtype=torch.long)
recon = fsq.range[idx]  # same device as fsq

fig, axes = plt.subplots(2, 1, figsize=(6, 7), sharex=False)

# Panel 1: quantization in [-1, 1]
ax = axes[0]
ax.scatter(tanh_x.cpu().numpy(), recon.cpu().numpy(), alpha=0.65, s=22, edgecolors="none")
lims = (-1.05, 1.05)
ax.plot(lims, lims, "k--", lw=1, alpha=0.45, label="y = x")
ax.set_xlabel("tanh(input)")
ax.set_ylabel("FSQ codebook value")
ax.set_title("FSQ maps tanh(x) to nearest of {} levels".format(fsq.n_levels))
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", frameon=False)

# Panel 2: error distribution
ax = axes[1]
err = (tanh_x - recon).abs().cpu().numpy()
ax.hist(err, bins=20, color="C0", edgecolor="white", linewidth=0.6)
ax.set_xlabel("|tanh(input) − reconstructed|")
ax.set_ylabel("count")
ax.set_title("Quantization error (L1 in tanh space)")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()